<a href="https://colab.research.google.com/github/krishkumarwork3-beep/ML-assignment-2/blob/main/ML_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
customers = pd.read_csv("AWCustomers.csv")
sales     = pd.read_csv("AWSales.csv")

In [ ]:
print("AWCustomers shape:", customers.shape)
print("AWSales shape:", sales.shape)

df = pd.merge(customers, sales[["CustomerID", "BikeBuyer"]], on="CustomerID", how="inner")
print("Merged shape:", df.shape)

print("\nColumn overview:")
for col in df.columns:
    print(f"{col:22s} dtype={str(df[col].dtype):8s} "
          f"nulls={df[col].isnull().sum():5d} nunique={df[col].nunique()}")

df["BirthDate"] = pd.to_datetime(df["BirthDate"])
reference_date = pd.Timestamp("2017-03-06")
df["Age"] = (reference_date - df["BirthDate"]).dt.days // 365

AWCustomers shape: (18361, 24)
AWSales shape: (18355, 3)
Merged shape: (18361, 25)

Column overview:
CustomerID             dtype=int64    nulls=    0 nunique=18355
Title                  dtype=object   nulls=18260 nunique=5
FirstName              dtype=object   nulls=    0 nunique=670
MiddleName             dtype=object   nulls= 7789 nunique=44
LastName               dtype=object   nulls=    0 nunique=375
Suffix                 dtype=object   nulls=18358 nunique=1
AddressLine1           dtype=object   nulls=    0 nunique=12742
AddressLine2           dtype=object   nulls=18050 nunique=166
City                   dtype=object   nulls=    0 nunique=269
StateProvinceName      dtype=object   nulls=    0 nunique=54
CountryRegionName      dtype=object   nulls=    0 nunique=6
PostalCode             dtype=object   nulls=    0 nunique=323
PhoneNumber            dtype=object   nulls=    0 nunique=8836
BirthDate              dtype=object   nulls=    0 nunique=8230
Education              dtype=obje

In [ ]:
selected_features = [
    "CountryRegionName",
    "Education",
    "Occupation",
    "Gender",
    "MaritalStatus",
    "HomeOwnerFlag",
    "NumberCarsOwned",
    "NumberChildrenAtHome",
    "YearlyIncome",
    "Age",
    "BikeBuyer"
]

In [ ]:
df_selected = df[selected_features].copy()

print("\nNew DataFrame (selected attributes):")
print(df_selected.shape)
print(df_selected.head())

print("\nMissing values in selected DataFrame:")
print(df_selected.isnull().sum())


New DataFrame (selected attributes):
(18361, 11)
  CountryRegionName        Education      Occupation Gender MaritalStatus  \
0         Australia        Bachelors        Clerical      M             M   
1            Canada  Partial College        Clerical      M             M   
2     United States        Bachelors        Clerical      F             S   
3    United Kingdom  Partial College  Skilled Manual      M             M   
4           Germany  Partial College  Skilled Manual      M             S   

   HomeOwnerFlag  NumberCarsOwned  NumberChildrenAtHome  YearlyIncome  Age  \
0              1                3                     0         81916   29   
1              1                2                     1         81076   44   
2              0                3                     0         86387   31   
3              1                2                     1         61481   39   
4              1                1                     0         51804   42   

   BikeBuyer  
0  

In [ ]:
data_type_summary = {
    "CountryRegionName":    {"MeasurementType": "Nominal", "Nature": "Discrete",
                              "Preprocessing": "One-hot / label encoding"},
    "Education":            {"MeasurementType": "Ordinal", "Nature": "Discrete",
                              "Preprocessing": "Ordinal encoding (Partial HS < HS < Partial College < Bachelors < Graduate)"},
    "Occupation":           {"MeasurementType": "Nominal", "Nature": "Discrete",
                              "Preprocessing": "One-hot encoding"},
    "Gender":               {"MeasurementType": "Nominal", "Nature": "Discrete",
                              "Preprocessing": "Binary encoding (M/F -> 0/1)"},
    "MaritalStatus":        {"MeasurementType": "Nominal", "Nature": "Discrete",
                              "Preprocessing": "Binary encoding (M/S -> 0/1)"},
    "HomeOwnerFlag":        {"MeasurementType": "Nominal (binary)", "Nature": "Discrete",
                              "Preprocessing": "Already 0/1, no change needed"},
    "NumberCarsOwned":      {"MeasurementType": "Ratio", "Nature": "Discrete",
                              "Preprocessing": "Keep numeric, optional scaling"},
    "NumberChildrenAtHome": {"MeasurementType": "Ratio", "Nature": "Discrete",
                              "Preprocessing": "Keep numeric, optional scaling"},
    "YearlyIncome":         {"MeasurementType": "Ratio", "Nature": "Continuous",
                              "Preprocessing": "Normalize / standardize (wide range)"},
    "Age":                  {"MeasurementType": "Ratio", "Nature": "Continuous",
                              "Preprocessing": "Normalize / standardize or bin into ranges"},
    "BikeBuyer":            {"MeasurementType": "Nominal (binary, TARGET)", "Nature": "Discrete",
                              "Preprocessing": "Keep as label (0/1), no transform needed"},
}

In [ ]:
type_df = pd.DataFrame(data_type_summary).T
type_df.index.name = "Attribute"
print("\nData Type / Preprocessing Summary:\n")
print(type_df.to_string())


df_selected.to_csv("AWCustomers_selected.csv", index=False)
type_df.to_csv("attribute_data_types.csv")

print("\nSaved: AWCustomers_selected.csv, attribute_data_types.csv")


Data Type / Preprocessing Summary:

                               MeasurementType      Nature                                                                Preprocessing
Attribute                                                                                                                              
CountryRegionName                      Nominal    Discrete                                                     One-hot / label encoding
Education                              Ordinal    Discrete  Ordinal encoding (Partial HS < HS < Partial College < Bachelors < Graduate)
Occupation                             Nominal    Discrete                                                             One-hot encoding
Gender                                 Nominal    Discrete                                                 Binary encoding (M/F -> 0/1)
MaritalStatus                          Nominal    Discrete                                                 Binary encoding (M/S -> 0/1)
HomeOwnerFl

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

0. LOAD THE SELECTED DATA FROM PART I  (full ~18000 rows)

In [ ]:
df = pd.read_csv("AWCustomers_selected.csv")
print("Loaded shape:", df.shape)
print(df.dtypes)

# Keep an untouched copy of the original selected data for reference
df_original = df.copy()

Loaded shape: (18361, 11)
CountryRegionName       object
Education               object
Occupation              object
Gender                  object
MaritalStatus           object
HomeOwnerFlag            int64
NumberCarsOwned          int64
NumberChildrenAtHome     int64
YearlyIncome             int64
Age                      int64
BikeBuyer                int64
dtype: object


(a) HANDLING NULL VALUES

In [ ]:
print("\nMissing values BEFORE handling:\n", df.isnull().sum())

# Numeric (Ratio/Interval) columns -> fill with median (robust to outliers)
numeric_cols = ["NumberCarsOwned", "NumberChildrenAtHome", "YearlyIncome", "Age"]
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical (Nominal/Ordinal) columns -> fill with mode (most frequent value)
categorical_cols = ["CountryRegionName", "Education", "Occupation",
                     "Gender", "MaritalStatus", "HomeOwnerFlag"]
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values AFTER handling:\n", df.isnull().sum())


Missing values BEFORE handling:
 CountryRegionName       0
Education               0
Occupation              0
Gender                  0
MaritalStatus           0
HomeOwnerFlag           0
NumberCarsOwned         0
NumberChildrenAtHome    0
YearlyIncome            0
Age                     0
BikeBuyer               0
dtype: int64

Missing values AFTER handling:
 CountryRegionName       0
Education               0
Occupation              0
Gender                  0
MaritalStatus           0
HomeOwnerFlag           0
NumberCarsOwned         0
NumberChildrenAtHome    0
YearlyIncome            0
Age                     0
BikeBuyer               0
dtype: int64


(b) NORMALIZATION  (Min-Max Scaling -> range [0, 1])

In [ ]:
# Applied to continuous/ratio attributes so all numeric ranges are comparable
minmax_cols = ["YearlyIncome", "Age", "NumberCarsOwned", "NumberChildrenAtHome"]

minmax_scaler = MinMaxScaler()
df_normalized = df.copy()
df_normalized[[c + "_norm" for c in minmax_cols]] = minmax_scaler.fit_transform(df[minmax_cols])

print("\nNormalized (Min-Max) columns sample:")
print(df_normalized[[c + "_norm" for c in minmax_cols]].head())


Normalized (Min-Max) columns sample:
   YearlyIncome_norm  Age_norm  NumberCarsOwned_norm  \
0           0.496842  0.185714                   0.6   
1           0.489453  0.400000                   0.4   
2           0.536172  0.214286                   0.6   
3           0.317083  0.328571                   0.4   
4           0.231958  0.371429                   0.2   

   NumberChildrenAtHome_norm  
0                   0.000000  
1                   0.333333  
2                   0.000000  
3                   0.333333  
4                   0.000000  


(c) DISCRETIZATION (BINNING)

In [ ]:
# --- Age binning ---
age_bins   = [0, 25, 35, 45, 55, 65, 100]
age_labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df_normalized["Age_binned"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, right=False)

# --- YearlyIncome binning (using quantile-based / equal-frequency binning) ---
df_normalized["Income_binned"] = pd.qcut(df["YearlyIncome"], q=4,
                                          labels=["Low", "Medium", "High", "Very High"])

print("\nAge_binned distribution:\n", df_normalized["Age_binned"].value_counts())
print("\nIncome_binned distribution:\n", df_normalized["Income_binned"].value_counts())



Age_binned distribution:
 Age_binned
25-34    6278
35-44    4697
<25      3748
45-54    2583
55-64     953
65+       102
Name: count, dtype: int64

Income_binned distribution:
 Income_binned
Low          4591
Medium       4590
High         4590
Very High    4590
Name: count, dtype: int64


(d) STANDARDIZATION  (Z-score -> mean 0, std 1)

In [ ]:
standard_cols = ["YearlyIncome", "Age", "NumberCarsOwned", "NumberChildrenAtHome"]

std_scaler = StandardScaler()
df_normalized[[c + "_zscore" for c in standard_cols]] = std_scaler.fit_transform(df[standard_cols])

print("\nStandardized (Z-score) columns sample:")
print(df_normalized[[c + "_zscore" for c in standard_cols]].head())


Standardized (Z-score) columns sample:
   YearlyIncome_zscore  Age_zscore  NumberCarsOwned_zscore  \
0             0.298555   -0.498349                1.892524   
1             0.271180    0.833426                0.798389   
2             0.444261   -0.320779                1.892524   
3            -0.367401    0.389501                0.798389   
4            -0.682765    0.655856               -0.295746   

   NumberChildrenAtHome_zscore  
0                    -0.594371  
1                     1.163279  
2                    -0.594371  
3                     1.163279  
4                    -0.594371  


(e) BINARIZATION (ONE-HOT ENCODING)

In [ ]:
# Nominal categorical attributes -> one-hot encode
nominal_cols = ["CountryRegionName", "Occupation", "Gender", "MaritalStatus"]

df_encoded = pd.get_dummies(df_normalized, columns=nominal_cols, prefix=nominal_cols)

# Ordinal attribute (Education) -> encode with meaningful order, NOT one-hot
education_order = {
    "Partial High School": 0,
    "High School": 1,
    "Partial College": 2,
    "Bachelors": 3,
    "Graduate Degree": 4
}
df_encoded["Education_encoded"] = df["Education"].map(education_order)
# HomeOwnerFlag is already binary (0/1) -> no transformation needed

print("\nFinal encoded dataframe shape:", df_encoded.shape)
print("\nFinal columns:\n", df_encoded.columns.tolist())
print(df_encoded.head())

# SAVE FINAL PREPROCESSED DATASET
df_encoded.to_csv("AWCustomers_preprocessed.csv", index=False)
print("\nSaved: AWCustomers_preprocessed.csv  (rows:", df_encoded.shape[0], ")")


Final encoded dataframe shape: (18361, 33)

Final columns:
 ['Education', 'HomeOwnerFlag', 'NumberCarsOwned', 'NumberChildrenAtHome', 'YearlyIncome', 'Age', 'BikeBuyer', 'YearlyIncome_norm', 'Age_norm', 'NumberCarsOwned_norm', 'NumberChildrenAtHome_norm', 'Age_binned', 'Income_binned', 'YearlyIncome_zscore', 'Age_zscore', 'NumberCarsOwned_zscore', 'NumberChildrenAtHome_zscore', 'CountryRegionName_Australia', 'CountryRegionName_Canada', 'CountryRegionName_France', 'CountryRegionName_Germany', 'CountryRegionName_United Kingdom', 'CountryRegionName_United States', 'Occupation_Clerical', 'Occupation_Management', 'Occupation_Manual', 'Occupation_Professional', 'Occupation_Skilled Manual', 'Gender_F', 'Gender_M', 'MaritalStatus_M', 'MaritalStatus_S', 'Education_encoded']
         Education  HomeOwnerFlag  NumberCarsOwned  NumberChildrenAtHome  \
0        Bachelors              1                3                     0   
1  Partial College              1                2                     